# 03 — Modeling & Evaluation

This notebook:
1. Trains Logistic Regression, Random Forest, and XGBoost
2. Evaluates each with stratified cross-validation (ROC-AUC)
3. Compares models and selects the best
4. Generates confusion matrices and ROC curves
5. Saves the best model to `models/model.pkl`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_cleaned_data
from src.features import engineer_features
from src.train import get_models, train_all_models, select_best_model, save_model
from src.evaluate import evaluate_model, find_optimal_threshold
from src.utils import split_data, setup_logging

setup_logging()
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Prepare Data

In [ ]:
df = load_cleaned_data()
df = engineer_features(df)
X_train, X_test, y_train, y_test = split_data(df)
print(f'Training: {X_train.shape}, Test: {X_test.shape}')
print(f'Default rate — Train: {y_train.mean():.3f}, Test: {y_test.mean():.3f}')

## 2. Train All Models

In [ ]:
fitted_models, cv_results = train_all_models(X_train, y_train, cv=5)

# Display CV results
cv_df = pd.DataFrame(cv_results).T
cv_df.columns = ['Mean AUC', 'Std AUC']
cv_df = cv_df.sort_values('Mean AUC', ascending=False)
print(cv_df)

In [ ]:
# Visualise CV results
fig, ax = plt.subplots(figsize=(8, 4))
cv_df['Mean AUC'].plot(kind='barh', ax=ax, xerr=cv_df['Std AUC'],
                       color=['#e74c3c', '#3498db', '#2ecc71'][:len(cv_df)],
                       edgecolor='black')
ax.set_title('Cross-Validated ROC-AUC Comparison')
ax.set_xlabel('ROC-AUC')
plt.tight_layout()
plt.show()

## 3. Evaluate Each Model on Test Set

In [ ]:
all_metrics = {}
for name, model in fitted_models.items():
    print(f'\n{"=" * 50}')
    print(f'  {name}')
    print(f'{"=" * 50}')
    metrics = evaluate_model(model, X_test, y_test, model_name=name)
    all_metrics[name] = metrics
    print(f'  ROC-AUC: {metrics["roc_auc"]:.4f}')
    print(f'  F1:      {metrics["f1_score"]:.4f}')
    print(f'  AP:      {metrics["average_precision"]:.4f}')

## 4. Select & Save Best Model

In [ ]:
best_name, best_model = select_best_model(cv_results, fitted_models)
save_model(best_model)
print(f'\n✅ Best model saved: {best_name}')

## 5. Threshold Optimization

In [ ]:
y_proba = best_model.predict_proba(X_test)[:, 1]

# Find optimal threshold (F1)
opt_threshold_f1 = find_optimal_threshold(y_test.values, y_proba, strategy='f1')
print(f'Optimal F1 threshold: {opt_threshold_f1:.4f}')

# Find threshold for >= 80% recall
opt_threshold_recall = find_optimal_threshold(y_test.values, y_proba, strategy='recall_80')
print(f'Threshold for >=80%% recall: {opt_threshold_recall:.4f}')

### Trade-off Analysis

- **F1-optimised threshold** balances precision and recall — good default.
- **Recall ≥ 80% threshold** is more conservative: catches more defaults but flags more false positives.
- In a **high-stakes lending** scenario, the recall-focused threshold is preferred to minimise bad loans.
- Choose based on the **cost ratio** of false negatives (missed defaults) vs. false positives (lost revenue from rejected customers).